# Project 9 - Favorita Store Sales Forecasting

## Notebook 01: Exploratory Data Analysis

**Goal.** Forecast daily unit sales for 1,782 store-family series across 54 Corporacion Favorita supermarkets in Ecuador over a 16-day horizon (2017-08-16 to 2017-08-31).

**Source.** Kaggle competition `store-sales-time-series-forecasting` (Corporacion Favorita, Ecuador). Train spans 2013-01-01 to 2017-08-15.

**Target.** `sales` (continuous, non-negative). Evaluation metric: Root Mean Squared Logarithmic Error (RMSLE).

This notebook covers: data loading, schema inspection, missingness, target distribution, hierarchical structure (store, family, cluster), exogenous regressors (oil, holidays, transactions), seasonality decomposition, autocorrelation, and a markdown summary at the end.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data').resolve()
print('DATA_DIR:', DATA_DIR)
print('Files:', sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Load all CSVs

`train.csv` is the main file (3M+ rows). The four side tables (`oil`, `holidays_events`, `transactions`, `stores`) are small. We parse `date` columns at load time.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA_DIR / 'stores.csv')
oil = pd.read_csv(DATA_DIR / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['date'])

print('train  ', train.shape, 'date range', train['date'].min().date(), '->', train['date'].max().date())
print('test   ', test.shape, 'date range', test['date'].min().date(), '->', test['date'].max().date())
print('stores ', stores.shape)
print('oil    ', oil.shape, 'date range', oil['date'].min().date(), '->', oil['date'].max().date())
print('holid  ', holidays.shape, 'date range', holidays['date'].min().date(), '->', holidays['date'].max().date())
print('trans  ', transactions.shape)

## 2. Schema and missingness

In [ ]:
def schema(df, name):
    s = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(),
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().mean() * 100).round(2),
    })
    print(f'--- {name} ---')
    print(s)
    print()

for df, name in [(train, 'train'), (test, 'test'), (stores, 'stores'),
                  (oil, 'oil'), (holidays, 'holidays'), (transactions, 'transactions')]:
    schema(df, name)

In [ ]:
train.head(3)

## 3. Hierarchy: stores and families

54 stores across 22 cities, 16 states, with five store types (A-E) and 17 clusters. 33 product families.

In [ ]:
print('Stores by type:')
print(stores['type'].value_counts())
print('\nStores by cluster:')
print(stores['cluster'].value_counts().sort_index())
print('\nStores by city (top 10):')
print(stores['city'].value_counts().head(10))
print('\nProduct families:')
print(sorted(train['family'].unique()))

## 4. Target distribution

`sales` is heavily right-skewed. The Kaggle metric is RMSLE which uses log1p(sales), so we inspect both raw and log scales.

In [ ]:
y = train['sales']
print('sales summary:')
print(y.describe().round(2))
print(f'\nZero-sales rows: {(y == 0).sum():,} ({(y == 0).mean()*100:.2f}%)')
print(f'Total rows: {len(train):,}')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.clip(y, 0, 200), bins=80, color='steelblue', edgecolor='white')
ax[0].set_title('sales (clipped at 200) - histogram')
ax[0].set_xlabel('sales')
ax[1].hist(np.log1p(y), bins=80, color='salmon', edgecolor='white')
ax[1].set_title('log1p(sales) - histogram')
ax[1].set_xlabel('log1p(sales)')
plt.tight_layout()
plt.show()

## 5. Aggregate sales over time (country level)

Sum sales across all stores and families per day. Look for trend, weekly seasonality, holiday spikes, and the April 2016 earthquake disruption.

In [ ]:
daily = train.groupby('date')['sales'].sum().reset_index()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily['date'], daily['sales'], color='steelblue', linewidth=0.7)
ax.axvline(pd.Timestamp('2016-04-16'), color='red', linestyle='--', alpha=0.5, label='Pedernales earthquake')
ax.set_title('Total daily sales across all stores and families (2013-2017)')
ax.set_xlabel('date')
ax.set_ylabel('total sales')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Weekly seasonality and day-of-week effect

In [ ]:
train['dow'] = train['date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = train.groupby('dow')['sales'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(8, 4))
dow_avg.plot(kind='bar', color='teal', ax=ax)
ax.set_title('Average sales by day of week')
ax.set_ylabel('mean sales')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 7. Family-level breakdown

Some families dominate volume (GROCERY I, BEVERAGES, PRODUCE). Others are sparse and intermittent (BOOKS, BABY CARE, HARDWARE).

In [ ]:
fam_total = train.groupby('family')['sales'].sum().sort_values(ascending=False)
fam_zero_pct = (train.assign(zero=(train['sales'] == 0))
                .groupby('family')['zero'].mean().sort_values(ascending=False) * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
fam_total.plot(kind='barh', ax=axes[0], color='darkblue')
axes[0].set_title('Total sales by family (2013-2017)')
axes[0].invert_yaxis()
fam_zero_pct.plot(kind='barh', ax=axes[1], color='crimson')
axes[1].set_title('% zero-sales rows by family (intermittency proxy)')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Oil price as exogenous regressor

Ecuador's economy is oil-linked. The `dcoilwtico` series has ~9% missing days (weekends, holidays) which we forward-fill.

In [ ]:
oil = oil.set_index('date').resample('D').asfreq()
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
print('oil missing after ffill+bfill:', oil['dcoilwtico'].isna().sum())

fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.plot(oil.index, oil['dcoilwtico'], color='black', linewidth=0.8, label='WTI oil price')
ax1.set_ylabel('USD per barrel', color='black')
ax2 = ax1.twinx()
ax2.plot(daily['date'], daily['sales'], color='steelblue', alpha=0.5, linewidth=0.7, label='total daily sales')
ax2.set_ylabel('total sales', color='steelblue')
ax1.set_title('WTI oil price vs total Favorita daily sales (2013-2017)')
plt.tight_layout()
plt.show()

## 9. Holiday calendar

`holidays_events` mixes national, regional, and local holidays plus Transfer / Bridge / Work Day rows. Filter `transferred == False` and treat `type == 'Transfer'` as the effective holiday.

In [ ]:
print('Holiday types:')
print(holidays['type'].value_counts())
print('\nHoliday locales:')
print(holidays['locale'].value_counts())
print('\nTransferred flag:')
print(holidays['transferred'].value_counts())

## 10. Promotions

In [ ]:
print('onpromotion summary:')
print(train['onpromotion'].describe().round(2))
print(f'\nRows with onpromotion > 0: {(train["onpromotion"] > 0).sum():,} ({(train["onpromotion"] > 0).mean()*100:.1f}%)')

promo_year = train.assign(year=train['date'].dt.year).groupby('year')['onpromotion'].mean()
print('\nMean onpromotion per row by year:')
print(promo_year.round(2))

## 11. Autocorrelation of total daily sales

Look for weekly (lag 7) and annual (lag 365) seasonality.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

y_total = daily.set_index('date')['sales']
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_acf(y_total, lags=60, ax=axes[0])
axes[0].set_title('ACF of total daily sales (60 lags)')
plot_pacf(y_total, lags=60, ax=axes[1])
axes[1].set_title('PACF of total daily sales (60 lags)')
plt.tight_layout()
plt.show()

## 12. Single-series sanity check

Pick one store-family combination (store 1, family GROCERY I) and inspect its full series.

In [ ]:
sub = train[(train['store_nbr'] == 1) & (train['family'] == 'GROCERY I')].sort_values('date')
print('store 1, family GROCERY I')
print(sub.describe().round(2))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(sub['date'], sub['sales'], color='steelblue', linewidth=0.6)
ax.set_title('Store 1, family GROCERY I - daily sales')
ax.set_xlabel('date')
ax.set_ylabel('sales')
plt.tight_layout()
plt.show()

## 13. Findings summary

**Dataset.**
- 3,000,888 train rows across 54 stores x 33 families x 1,684 days (2013-01-01 to 2017-08-15).
- Test horizon is 16 days (2017-08-16 to 2017-08-31), 28,512 rows.
- Side tables: stores (54), oil (1,218 days, 9% missing), holidays (350 events), transactions (83,488 store-days).

**Target distribution.**
- `sales` is heavily right-skewed. A non-trivial share of rows is exactly zero (intermittent demand on small families).
- The Kaggle metric is RMSLE, which is well-behaved on log1p(sales) and tolerant to the zero-mass.

**Hierarchy and intermittency.**
- 1,782 active store-family series. A handful of families (BOOKS, BABY CARE, HARDWARE) have >50% zero-sales rows and need Croston-style intermittent treatment, not standard regression.
- Volume is concentrated in GROCERY I, BEVERAGES, and PRODUCE.

**Seasonality.**
- Strong weekly seasonality (weekend uplift in most families).
- Annual seasonality with December peak (Christmas) and a smaller May peak (Mother's Day).
- The Pedernales earthquake (16 April 2016) creates a multi-week disruption visible in the country-level total.

**Exogenous drivers.**
- Oil price negatively correlates with retail demand on monthly aggregates (Ecuador macro effect), but the daily relationship is noisy.
- Holiday calendar mixes national, regional (state), and local (city) events. Transfer rows carry the effective holiday date.
- `onpromotion` is the count of items on promotion in that store-family-day; correlates positively with same-day sales.

**Implications for modelling.**
- Baseline: per-store-family Prophet or SARIMA with weekly + yearly seasonality, holiday regressor matrix, and oil price as additional regressor. Loop is roughly 1,782 fits at 1-2 seconds each.
- Advanced: single LightGBM model trained on the full panel, with lag features (1, 7, 14, 28), rolling means (7d, 28d), holiday flags (national, regional, local), oil price, store and family categorical encodings, and a quantile-regression head for prediction intervals.
- Validation: time-based holdout (last 16 days of train as pseudo-test) to mirror the Kaggle public leaderboard window.
- Out-of-scope for this scaffold: hierarchical reconciliation (MinT), DeepAR / TFT, intermittent-demand specialised models. These are documented in the manuscript Discussion.